# YOLOv8 Sign Detector — MTSD Training

Trains a binary YOLOv8n sign detector on the Mapillary Traffic Sign Dataset (MTSD).

**Before running:**
- Add dataset `zeuss2k3/mapillary-traffic-sign-dataset` via the Data panel
- Enable GPU accelerator (Settings → Accelerator → GPU T4)
- Enable Internet (Settings → Internet → On)

**Output:** `yolov8n_mtsd_best.pt` in the Output panel — download after training.

In [ ]:
# ── Cell 1: Environment ───────────────────────────────────────────────────────
import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

!pip install ultralytics -q
import ultralytics
print(f'Ultralytics: {ultralytics.__version__}')

In [ ]:
# ── Cell 2: Locate dataset + resolve paths ────────────────────────────────────
import os
from pathlib import Path

WORK = Path('/kaggle/working')

# Find dataset root
DS_ROOT = None
for p in Path('/kaggle/input').rglob('full_ds'):
    DS_ROOT = p.parent
    break
if DS_ROOT is None:
    for p in Path('/kaggle/input').rglob('mtsd_v2_fully_annotated'):
        DS_ROOT = p.parent.parent
        break
if DS_ROOT is None:
    raise RuntimeError('Dataset not found — add zeuss2k3/mapillary-traffic-sign-dataset via the Data panel')
print(f'Dataset root: {DS_ROOT}')

# Resolve sub-paths (matches zeuss2k3 Kaggle dataset layout)
ANN_DIR    = DS_ROOT / 'mtsd_v2_fully_annotated_annotation.zip' / 'mtsd_v2_fully_annotated' / 'annotations'
SPLITS_DIR = DS_ROOT / 'mtsd_v2_fully_annotated_annotation.zip' / 'mtsd_v2_fully_annotated' / 'splits'
TRAIN_IMGS = DS_ROOT / 'mtsd_fully_annotated_train_images' / 'images'
VAL_IMGS   = DS_ROOT / 'mtsd_v2_fully_annotated_images.val.zip' / 'images'

for name, p in [('ANN_DIR', ANN_DIR), ('SPLITS_DIR', SPLITS_DIR),
                ('TRAIN_IMGS', TRAIN_IMGS), ('VAL_IMGS', VAL_IMGS)]:
    status = '✓' if p.exists() else '✗ NOT FOUND'
    print(f'  {status}  {name}: {p}')

print(f'\n  train images : {len(list(TRAIN_IMGS.glob("*.jpg"))):,}')
print(f'  val images   : {len(list(VAL_IMGS.glob("*.jpg"))):,}')
print(f'  annotations  : {len(list(ANN_DIR.glob("*.json"))):,}')

In [ ]:
# ── Cell 3: Prepare YOLO-format label files ───────────────────────────────────
import json
from tqdm.notebook import tqdm

LABEL_ROOT = WORK / 'labels'


def boxes_to_yolo(ann):
    """Convert MTSD annotation to YOLO lines. Returns None for panoramic images."""
    if ann.get('ispano', False):
        return None
    img_w, img_h = ann['width'], ann['height']
    lines = []
    for obj in ann.get('objects', []):
        props = obj.get('properties', {})
        if props.get('out-of-frame') or props.get('ambiguous'):
            continue
        b = obj['bbox']
        xmin = max(0.0,          b['xmin'])
        ymin = max(0.0,          b['ymin'])
        xmax = min(float(img_w), b['xmax'])
        ymax = min(float(img_h), b['ymax'])
        if xmax <= xmin or ymax <= ymin:
            continue
        cx = (xmin + xmax) / 2 / img_w
        cy = (ymin + ymax) / 2 / img_h
        w  = (xmax - xmin) / img_w
        h  = (ymax - ymin) / img_h
        lines.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    return lines


def prepare_split(split_name, images_dir):
    keys = (SPLITS_DIR / f'{split_name}.txt').read_text().splitlines()
    keys = [k.strip() for k in keys if k.strip()]

    label_dir = LABEL_ROOT / split_name
    label_dir.mkdir(parents=True, exist_ok=True)

    stats = dict(written=0, boxes=0, negatives=0, skipped=0)
    for key in tqdm(keys, desc=split_name):
        img_path = images_dir / f'{key}.jpg'
        ann_path = ANN_DIR    / f'{key}.json'
        if not img_path.exists() or not ann_path.exists():
            stats['skipped'] += 1
            continue
        ann   = json.loads(ann_path.read_text())
        lines = boxes_to_yolo(ann)
        if lines is None:
            stats['skipped'] += 1
            continue
        (label_dir / f'{key}.txt').write_text('\n'.join(lines) + ('\n' if lines else ''))
        stats['written']   += 1
        stats['boxes']     += len(lines)
        stats['negatives'] += int(not lines)
    return stats


print('Preparing train split...')
s = prepare_split('train', TRAIN_IMGS)
print(f"  written={s['written']:,}  boxes={s['boxes']:,}  negatives={s['negatives']:,}  skipped={s['skipped']:,}")

print('Preparing val split...')
s = prepare_split('val', VAL_IMGS)
print(f"  written={s['written']:,}  boxes={s['boxes']:,}  negatives={s['negatives']:,}  skipped={s['skipped']:,}")

In [ ]:
# ── Cell 4: Write dataset.yaml ────────────────────────────────────────────────
# Point directly at the real image directories (no symlinks needed —
# the monkey-patch in Cell 5 handles label resolution).
import yaml

DATASET_YAML = WORK / 'dataset.yaml'
with open(DATASET_YAML, 'w') as f:
    yaml.dump({
        'path' : str(WORK),
        'train': str(TRAIN_IMGS),
        'val'  : str(VAL_IMGS),
        'nc'   : 1,
        'names': ['sign'],
    }, f, default_flow_style=False, sort_keys=False)

print(DATASET_YAML.read_text())

In [ ]:
# ── Cell 5: Patch ultralytics label resolution ────────────────────────────────
# ultralytics resolves symlinks before substituting /images/ -> /labels/,
# so it would look for labels in the read-only /kaggle/input directory.
# We intercept the label path lookup and redirect it to our written label files.

import importlib
from pathlib import Path

# Build a fast stem -> label path lookup
label_lookup = {}
for split in ['train', 'val']:
    for f in (WORK / 'labels' / split).glob('*.txt'):
        label_lookup[f.stem] = str(f)
print(f'Label lookup: {len(label_lookup):,} entries')

import ultralytics.data.utils as _du
_orig = _du.img2label_paths

def _patched_img2label_paths(img_paths):
    result = []
    for x in img_paths:
        stem = Path(x).stem
        result.append(label_lookup.get(stem, _orig([x])[0]))
    return result

_du.img2label_paths = _patched_img2label_paths
for _mod in ['ultralytics.data.dataset', 'ultralytics.data.build', 'ultralytics.data']:
    try:
        m = importlib.import_module(_mod)
        if hasattr(m, 'img2label_paths'):
            setattr(m, 'img2label_paths', _patched_img2label_paths)
            print(f'  patched {_mod}')
    except Exception:
        pass

print('Done ✓')

In [ ]:
# ── Cell 6: Train YOLOv8n ─────────────────────────────────────────────────────
from ultralytics import YOLO

device = '0' if torch.cuda.is_available() else 'cpu'
print(f'Training on device: {device}')

model = YOLO('yolov8n.pt')

results = model.train(
    data        = str(DATASET_YAML),
    epochs      = 25,
    imgsz       = 640,
    batch       = 32,
    device      = device,
    project     = str(WORK / 'runs'),
    name        = 'yolov8n_mtsd',
    exist_ok    = True,
    workers     = 4,
    degrees     = 10,
    scale       = 0.5,
    fliplr      = 0.5,
    mosaic      = 1.0,
    patience    = 15,
    verbose     = True,
    plots       = True,
    save_period = 5,
)

print(f"\nBest mAP50: {results.results_dict.get('metrics/mAP50(B)', 'N/A')}")

In [ ]:
# ── Cell 7: Save output + show curves ────────────────────────────────────────
import shutil
from IPython.display import Image as IPImage, display

run_dir  = WORK / 'runs' / 'yolov8n_mtsd'
best_src = run_dir / 'weights' / 'best.pt'
best_dst = WORK / 'yolov8n_mtsd_best.pt'

if best_src.exists():
    shutil.copy2(best_src, best_dst)
    print(f'Saved: {best_dst}  ({best_dst.stat().st_size / 1e6:.1f} MB)')
    print('Download from the Output panel →')
else:
    print(f'best.pt not found — available weights:')
    for f in sorted(run_dir.rglob('*.pt')):
        print(f'  {f}')

# Copy results files to /kaggle/working/ so they're accessible via the API
results_files = [
    'results.png',
    'results.csv',
    'args.yaml',
    'confusion_matrix_normalized.png',
    'val_batch0_pred.jpg',
    'val_batch1_pred.jpg',
]
for fname in results_files:
    src = run_dir / fname
    if src.exists():
        shutil.copy2(src, WORK / fname)
        print(f'Copied: {fname}')
    else:
        print(f'Not found (skipped): {fname}')

for plot in ['results.png', 'confusion_matrix_normalized.png', 'val_batch0_pred.jpg']:
    p = run_dir / plot
    if p.exists():
        print(f'\n--- {plot} ---')
        display(IPImage(str(p)))